Herarchical RL

In [ ]:
import pandas as pd

df = pd.read_csv("../data/gsea_cleaned.csv")
df.head()

In [ ]:
import math
import random
from collections import deque, namedtuple
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

Transition = namedtuple('Transition', ('state', 'action', 'reward', 'next_state', 'done', 'goal'))

def set_seed(s=42):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)

class GenePathwayEnv:
    def __init__(self, n_genes=10, n_pathways=3, max_steps=50):
        self.n_genes = n_genes
        self.n_pathways = n_pathways
        self.max_steps = max_steps
        self.reset()

    def reset(self):
        self.state = np.random.uniform(0, 1, size=(self.n_genes,)).astype(np.float32)
        self.step_count = 0
        self.target = np.zeros(self.n_genes, dtype=np.float32)
        for p in range(self.n_pathways):
            idx = np.random.choice(self.n_genes, size=max(1, self.n_genes//(self.n_pathways+1)), replace=False)
            self.target[idx] = 1.0
        return self.state.copy()

    def step(self, action):
        self.step_count += 1
        action = np.clip(action, -1, 1)
        self.state = np.clip(self.state + action, 0.0, 1.0)
        dist = np.linalg.norm(self.state - self.target)
        reward = -dist
        done = self.step_count >= self.max_steps or dist < 1e-2
        return self.state.copy(), float(reward), done, {}

class MLP(nn.Module):
    def __init__(self, in_dim, out_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, out_dim)
        )

    def forward(self, x):
        return self.net(x)

class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buf = deque(maxlen=capacity)

    def push(self, *args):
        self.buf.append(Transition(*args))

    def sample(self, batch_size):
        batch = random.sample(self.buf, batch_size)
        return Transition(*zip(*batch))

    def __len__(self):
        return len(self.buf)

class Manager:
    def __init__(self, state_dim, goal_dim, lr=1e-3):
        self.net = MLP(state_dim, goal_dim)
        self.opt = optim.Adam(self.net.parameters(), lr=lr)

    def select_goal(self, state, deterministic=False):
        s = torch.tensor(state).float().unsqueeze(0)
        g = self.net(s).detach().numpy()[0]
        if not deterministic and random.random() < 0.2:
            g = np.random.normal(size=g.shape)
        return g

    def update(self, batch):
        if len(batch.state) == 0:
            return
        s = torch.tensor(np.array(batch.state)).float()
        g = torch.tensor(np.array(batch.goal)).float()
        r = torch.tensor(np.array(batch.reward)).float().unsqueeze(1)
        pred = self.net(s)
        loss = ((pred - g)**2).mean() - r.mean()
        self.opt.zero_grad()
        loss.backward()
        self.opt.step()

class Worker:
    def __init__(self, state_dim, goal_dim, action_dim, lr=1e-3):
        self.net = MLP(state_dim + goal_dim, action_dim)
        self.opt = optim.Adam(self.net.parameters(), lr=lr)

    def select_action(self, state, goal, deterministic=False):
        s = np.concatenate([state, goal], axis=0)
        s = torch.tensor(s).float().unsqueeze(0)
        a = self.net(s).detach().numpy()[0]
        if not deterministic and random.random() < 0.1:
            a += np.random.normal(scale=0.1, size=a.shape)
        return np.tanh(a)

    def update(self, batch):
        if len(batch.state) == 0:
            return
        s = np.array(batch.state)
        g = np.array(batch.goal)
        x = torch.tensor(np.concatenate([s, g], axis=1)).float()
        a = torch.tensor(np.array(batch.action)).float()
        r = torch.tensor(np.array(batch.reward)).float().unsqueeze(1)
        pred = self.net(x)
        loss = ((pred - a)**2 * (1.0 + r.unsqueeze(1))).mean()
        self.opt.zero_grad()
        loss.backward()
        self.opt.step()

def train(num_episodes=200, manager_horizon=5):
    set_seed()
    env = GenePathwayEnv(n_genes=16, n_pathways=4, max_steps=60)
    state_dim = env.n_genes
    goal_dim = 8
    action_dim = env.n_genes
    manager = Manager(state_dim, goal_dim)
    worker = Worker(state_dim, goal_dim, action_dim)
    buffer = ReplayBuffer(20000)
    for ep in range(num_episodes):
        s = env.reset()
        done = False
        total_r = 0.0
        step = 0
        current_goal = manager.select_goal(s)
        while not done:
            if step % manager_horizon == 0:
                current_goal = manager.select_goal(s)
            a = worker.select_action(s, current_goal)
            ns, r, done, _ = env.step(a)
            buffer.push(s, a, r, ns, done, current_goal)
            total_r += r
            s = ns
            step += 1
            if len(buffer) > 256:
                batch = buffer.sample(128)
                worker.update(batch)
                manager.update(batch)
        if (ep + 1) % 10 == 0:
            print(f"ep={ep+1} total_r={total_r:.2f} steps={step}")
    return manager, worker, env

def test(manager, worker, env, episodes=5):
    for ep in range(episodes):
        s = env.reset()
        done = False
        total = 0.0
        step = 0
        while not done and step < env.max_steps:
            if step % 5 == 0:
                g = manager.select_goal(s, deterministic=True)
            a = worker.select_action(s, g, deterministic=True)
            s, r, done, _ = env.step(a)
            total += r
            step += 1
        print(f"test_ep={ep+1} total={total:.2f} steps={step}")

if __name__ == '__main__':
    m, w, e = train(num_episodes=80)
    test(m, w, e, episodes=6)
